In [ ]:
import pandas as pd
import numpy as np
import os
import gc
import math
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
# パス設定
INPUT_DIR = '.'
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_spectrogram_2dcnn_pid.csv'

# ハイパーパラメータ
BATCH_SIZE = 64
N_FOLDS = 10          # 時間短縮のため5分割推奨（精度追求なら10）
EPOCHS = 20
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 特徴量エンジニアリング (PID Features)
# ==============================================================================
def add_pid_features(df):
    """
    PID特徴量 + 時間重み付け
    """
    df_eng = df.copy()
    
    # 除外カラム
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    print(f"Generating PID features for {len(base_cols)} base columns...")
    
    # 時間重み付け
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    # Groupbyオブジェクト
    grouped = df_eng.groupby('sample_id')
    
    for col in base_cols:
        # P (Proportional)
        df_eng[f'{col}_weighted'] = df_eng[col] * gain
        
        # D (Derivative)
        col_diff1 = grouped[col].diff().fillna(0)
        df_eng[f'{col}_diff1'] = col_diff1 * gain
        
        # I (Integral)
        df_eng[f'{col}_cumsum'] = grouped[col].cumsum()
        
        # Lag
        df_eng[f'{col}_lag1'] = grouped[col].shift(1).fillna(0)
        
    return df_eng

print(">>> Loading Data & Engineering PID Features...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

# 特徴量生成
train = add_pid_features(train)
test = add_pid_features(test)

target_col = 'lever'
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
feature_cols = [c for c in train_feats if c in test_feats]

print(f"Total Input Features (Channels): {len(feature_cols)}")

# スケーリング (RobustScaler)
scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dynamic MAX_LEN & Dataset
# ==============================================================================
# 動的に最大長を取得
max_seq_len_train = train.groupby('sample_id').size().max()
max_seq_len_test = test.groupby('sample_id').size().max()
MAX_RAW_LEN = max(max_seq_len_train, max_seq_len_test)

# CNNのPooling用に32の倍数にする
MAX_LEN = math.ceil(MAX_RAW_LEN / 32) * 32
print(f"Dynamic MAX_LEN set to: {MAX_LEN}")

class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=MAX_LEN):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {
            'x': torch.tensor(x_pad), 
            'mask': torch.tensor(mask), 
            'id': sample_id,
            'seq_len': seq_len
        }
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
            
        return ret

# ==============================================================================
# 4. Model Definition: Spectrogram 2D-CNN
# ==============================================================================
class Spectrogram2DCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        
        # CNN Backbone
        # 入力: [Batch, 1, Features, Time]
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)), 
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)), 
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # 特徴量軸(H)と時間軸(W)の両方を (1, 1) に圧縮して
            # 「このサンプル全体の静的な特徴」を抽出する
            nn.AdaptiveAvgPool2d((1, 1)) 
        )
        
        # LSTM
        self.lstm = nn.LSTM(
            input_size=input_dim + 128, # 元の時系列データ + CNNが抽出した静的特徴
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: [Batch, Time, Features]
        bs, seq_len, n_feats = x.shape
        
        # --- 画像化 (CNN入力用) ---
        # [Batch, 1, Features, Time]
        img = x.permute(0, 2, 1).unsqueeze(1) 
        
        # CNN特徴抽出 -> [Batch, 128, 1, 1]
        feat_img = self.features(img)
        feat_img = feat_img.view(bs, 128)
        
        # LSTM入力用に、CNN特徴を全時刻にコピー
        feat_repeated = feat_img.unsqueeze(1).repeat(1, seq_len, 1) # [Batch, Time, 128]
        
        # 元の時系列データと結合
        x_concat = torch.cat([x, feat_repeated], dim=2)
        
        # LSTM
        lstm_out, _ = self.lstm(x_concat)
        output = self.head(lstm_out)
        
        return output.squeeze(-1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
# 提出用IDマッピング (重要)
test_sample_map = test.groupby('sample_id')['id'].apply(list).to_dict()
submission_dict = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold Cross Validation (Spectrogram 2D-CNN + PID)...")

# Test Dataset
test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, target_col, MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, target_col, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = Spectrogram2DCNN(input_dim=len(feature_cols), hidden_dim=64).to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_spec_pid_fold{fold}.pth"
    
    # Train
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")
        
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
    print(f"  >> Best Val: {best_loss:.5f}")
    
    # Inference (Fixed Logic)
    print("  Predicting Test Data...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    
    with torch.no_grad():
        for batch in test_loader:
            x = batch['x'].to(DEVICE)
            s_ids = batch['id']
            seq_lens = batch['seq_len'].numpy()
            
            preds = model(x).cpu().numpy()
            
            for i, s_id in enumerate(s_ids):
                valid_len = seq_lens[i]
                # 有効な長さ分だけ取り出し、負の値をクリップ
                pred_seq = np.maximum(0, preds[i, :valid_len])
                
                # 正しいrow_idを取得
                row_ids = test_sample_map[s_id]
                
                # 長さチェックと補正
                if len(row_ids) != len(pred_seq):
                    min_len = min(len(row_ids), len(pred_seq))
                    row_ids = row_ids[:min_len]
                    pred_seq = pred_seq[:min_len]
                
                # アンサンブル用に加算
                for r_id, p_val in zip(row_ids, pred_seq):
                    submission_dict[r_id] = submission_dict.get(r_id, 0) + p_val

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# 6. Final Submission
# ==============================================================================
print("\n>>> Creating Final Submission...")

if len(submission_dict) > 0:
    results = []
    # test.csvの並び順で作成
    for r_id in test['id']:
        val = submission_dict.get(r_id, 0) / N_FOLDS
        results.append({'id': r_id, 'lever': val})

    df_sub = pd.DataFrame(results)
    df_sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"SUCCESS: {SUBMISSION_PATH} created.")
else:
    print("Error: No predictions generated.")

Using Device: cpu
>>> Loading Data & Engineering PID Features...
Generating PID features for 44 base columns...
Generating PID features for 44 base columns...
Total Input Features (Channels): 220
Dynamic MAX_LEN set to: 64

>>> Starting 10-Fold Cross Validation (Spectrogram 2D-CNN + PID)...

=== Fold 1/10 ===
  Ep  1 | Train: 1.3026 | Val: 1.1735
